# Support Vector Machines
## From Maximum-Margin Intuition to the Kernel Trick

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/svm_lesson.ipynb)

---

Support Vector Machines (SVMs) are among the most theoretically elegant learning algorithms ever devised. Developed by Vapnik and collaborators in the 1990s, they unify beautiful geometry, convex optimization, and functional analysis into a surprisingly practical tool — while providing *non-asymptotic* generalization guarantees grounded in the theory of VC dimension.

**Roadmap:**
1. **The geometry of margins** — why maximizing distance to the decision boundary leads to better generalization
2. **Hard-margin SVM** — the primal quadratic program, Lagrangian duality, and KKT conditions
3. **Soft-margin SVM** — slack variables and the regularization parameter $C$
4. **The kernel trick** — implicitly mapping to (possibly infinite-dimensional) feature spaces
5. **SVMs in practice** — hyperparameter tuning, multi-class extensions, and scikit-learn

> **Prerequisites:** Linear algebra (inner products, projections), multivariate calculus (gradients, Lagrange multipliers), Python/NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.svm import SVC, LinearSVC
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})
print('Libraries loaded.')

---
# Part 1: The Geometry of Classification

## 1.1 The Decision Boundary Problem

Consider a binary classification task: given training data $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $\mathbf{x}_i \in \mathbb{R}^d$ and labels $y_i \in \{-1, +1\}$, we want to find a **hyperplane** that separates the two classes.

A hyperplane in $\mathbb{R}^d$ is defined by a normal vector $\mathbf{w} \in \mathbb{R}^d$ and a bias $b \in \mathbb{R}$:
$$H = \{\mathbf{x} \in \mathbb{R}^d : \mathbf{w} \cdot \mathbf{x} + b = 0\}.$$

We classify a new point $\mathbf{x}$ as class $+1$ if $\mathbf{w} \cdot \mathbf{x} + b > 0$, and $-1$ otherwise.

**The problem:** When data is linearly separable, infinitely many hyperplanes separate the classes perfectly. The cell below shows three such hyperplanes — which one is best, and why?

In [ ]:
# Generate linearly separable 2-class data
rng = np.random.default_rng(7)
X_pos = rng.multivariate_normal([1.5, 1.5], [[0.35, 0.1], [0.1, 0.35]], size=20)
X_neg = rng.multivariate_normal([-1.5, -1.5], [[0.35, 0.1], [0.1, 0.35]], size=20)
X_sep = np.vstack([X_pos, X_neg])
y_sep = np.hstack([np.ones(20), -np.ones(20)])

# Three candidate decision boundaries
hyperplanes = [
    (np.array([1.0,  1.0]),  0.0,  '#e74c3c', 'Hyperplane A'),
    (np.array([1.0,  0.4]),  0.4,  '#3498db', 'Hyperplane B'),
    (np.array([0.7,  1.3]), -0.2,  '#2ecc71', 'Hyperplane C'),
]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#3498db', s=60, edgecolors='k',
           lw=0.5, label='+1 class', zorder=3)
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#e74c3c', s=60, marker='s',
           edgecolors='k', lw=0.5, label=r'$-$1 class', zorder=3)

xx = np.linspace(-3.5, 3.5, 200)
for (w, b, color, label) in hyperplanes:
    yy = (-w[0] * xx - b) / w[1]
    ax.plot(xx, yy, color=color, lw=2, label=label, alpha=0.85)

ax.set_xlim(-3.2, 3.2)
ax.set_ylim(-3.2, 3.2)
ax.set_xlabel(r'$x_1$', fontsize=13)
ax.set_ylabel(r'$x_2$', fontsize=13)
ax.set_title('Many hyperplanes separate the data\u2014which is best?', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
plt.tight_layout()
plt.show()

## 1.2 Maximizing the Margin

The **signed distance** from a point $\mathbf{x}_i$ to the hyperplane $\{\mathbf{w} \cdot \mathbf{x} + b = 0\}$ is
$$d_i = \frac{\mathbf{w} \cdot \mathbf{x}_i + b}{\|\mathbf{w}\|}.$$

For a correctly classified point with label $y_i$, we have $y_i d_i > 0$. We define the **geometric margin** of a classifier as twice the distance from the hyperplane to the *nearest* training point:
$$\text{margin} = 2 \cdot \min_i \frac{y_i(\mathbf{w} \cdot \mathbf{x}_i + b)}{\|\mathbf{w}\|}.
$$

The **canonical normalization** scales $(\mathbf{w}, b)$ so that the nearest points satisfy $|\mathbf{w} \cdot \mathbf{x}_i + b| = 1$. Under this convention the margin becomes $2/\|\mathbf{w}\|$.

**Key insight: support vectors.**  The training points that achieve the minimum distance $1/\|\mathbf{w}\|$ (i.e., those lying on the margin boundaries $\mathbf{w}\cdot\mathbf{x}+b = \pm 1$) are called **support vectors**. Every other training point could be removed from the dataset without changing the optimal hyperplane.

**Why maximize the margin?**  Intuitively, a wider margin provides a safer buffer against measurement noise and new test points near the boundary. This is formalized in VC theory: among all consistent linear classifiers, the one with maximum margin achieves the smallest upper bound on the generalization error (Vapnik–Chervonenkis theorem).

In [ ]:
# Fit the maximum-margin SVM (hard margin: large C)
clf_sep = SVC(kernel='linear', C=1e6)
clf_sep.fit(X_sep, y_sep)

w = clf_sep.coef_[0]
b = clf_sep.intercept_[0]
margin = 2.0 / np.linalg.norm(w)

fig, ax = plt.subplots(figsize=(7, 6))

# Background shading and margin strip
xx_bg = np.linspace(-3.5, 3.5, 300)
yy_bg = np.linspace(-3.5, 3.5, 300)
XX, YY = np.meshgrid(xx_bg, yy_bg)
Z = clf_sep.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)

ax.contourf(XX, YY, Z, levels=[-100, -1, 0, 1, 100],
            colors=['#fde8e8', '#fde8e8', '#e8f4fd', '#e8f4fd'], alpha=0.55)
ax.contour(XX, YY, Z, levels=[-1, 0, 1],
           colors=['#c0392b', 'k', '#2980b9'],
           linewidths=[1.5, 2.5, 1.5],
           linestyles=['--', '-', '--'])

# Data
ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#3498db', s=60, edgecolors='k',
           lw=0.5, label='+1 class', zorder=3)
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#e74c3c', s=60, marker='s',
           edgecolors='k', lw=0.5, label=r'$-$1 class', zorder=3)

# Highlight support vectors
sv = clf_sep.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1], s=220, facecolors='none',
           edgecolors='gold', linewidths=2.5, zorder=4, label='Support vectors')

# Margin width annotation
w_hat = w / np.linalg.norm(w)
pt_bd = np.array([0.0, -b / w[1]])          # a point on the decision boundary
pt1   = pt_bd + w_hat / np.linalg.norm(w)   # on the +1 margin line
pt2   = pt_bd - w_hat / np.linalg.norm(w)   # on the -1 margin line
ax.annotate('', xy=pt1, xytext=pt2,
            arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=2.0))
ax.text(pt_bd[0] + 0.12, pt_bd[1],
        f'margin = {margin:.2f}', color='darkgreen', fontsize=11)

ax.set_xlim(-3.2, 3.2)
ax.set_ylim(-3.2, 3.2)
ax.set_xlabel(r'$x_1$', fontsize=13)
ax.set_ylabel(r'$x_2$', fontsize=13)
ax.set_title(f'Maximum-margin hyperplane (margin = {margin:.2f})', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Weight vector  w = [{w[0]:.4f}, {w[1]:.4f}]')
print(f'Bias           b = {b:.4f}')
print(f'Margin             = {margin:.4f}')
print(f'Support vectors: {len(sv)}')

> **Exercise 1.1.** *(Identifying support vectors)*
>
> Consider the following 2-dimensional dataset:
> $$\mathbf{x}_1=(1,2)^\top,\; y_1=+1;\quad \mathbf{x}_2=(2,3)^\top,\; y_2=+1;\quad \mathbf{x}_3=(3,1)^\top,\; y_3=+1;$$
> $$\mathbf{x}_4=(-1,-1)^\top,\; y_4=-1;\quad \mathbf{x}_5=(-2,0)^\top,\; y_5=-1;\quad \mathbf{x}_6=(-1,-3)^\top,\; y_6=-1.$$
>
> **(a)** Sketch these points on paper. Using geometric intuition alone, identify which points are likely to be support vectors.
>
> **(b)** Fit an SVM with `SVC(kernel='linear', C=1e6)` and verify your guess by inspecting `clf.support_vectors_` and `clf.support_`.
>
> **(c)** Report the margin of the optimal hyperplane. Verify that $y_i(\mathbf{w}\cdot\mathbf{x}_i+b)=1$ for support vectors and $>1$ for non-support vectors.
>
> **(d)** What happens to the support vectors and the margin when you add the point $\mathbf{x}_7 = (0, 0)^\top$ with $y_7 = +1$?

---
# Part 2: Hard-Margin SVM — The Mathematics

## 2.1 The Primal Optimization Problem

We want to find $(\mathbf{w}, b)$ that (i) correctly classifies all training points and (ii) maximizes the margin $2/\|\mathbf{w}\|$. Under the canonical normalization, the conditions are:
$$y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1, \quad i = 1, \ldots, n.$$

Maximizing $2/\|\mathbf{w}\|$ is equivalent to minimizing $\|\mathbf{w}\|^2/2$. The **hard-margin SVM primal problem** is therefore:

$$\boxed{\min_{\mathbf{w},\, b}\; \frac{1}{2}\|\mathbf{w}\|^2 \qquad \text{subject to}\quad y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1,\quad i=1,\ldots,n.}$$

This is a **strictly convex quadratic program** (QP): the objective is a strictly convex quadratic function of $(\mathbf{w}, b)$ and all constraints are linear. It therefore has a *unique* global minimizer.

In [ ]:
# Inspect the hard-margin SVM solution analytically
clf_hard = SVC(kernel='linear', C=1e8)
clf_hard.fit(X_sep, y_sep)

w_h  = clf_hard.coef_[0]
b_h  = clf_hard.intercept_[0]
SVs  = clf_hard.support_vectors_
SV_y = y_sep[clf_hard.support_]

print('=== Hard-Margin SVM Solution ===')
print(f'  w = [{w_h[0]:.5f}, {w_h[1]:.5f}]')
print(f'  b = {b_h:.5f}')
print(f'  ||w|| = {np.linalg.norm(w_h):.5f}')
print(f'  Margin = 2/||w|| = {2/np.linalg.norm(w_h):.5f}')
print()
print(f'  Support vectors ({len(SVs)} total):')
for i, (sv, yi) in enumerate(zip(SVs, SV_y)):
    constraint_val = yi * (w_h @ sv + b_h)
    print(f'    SV{i+1}: x={sv.round(4)}, y={int(yi):+d},  '
          f'y(w·x+b) = {constraint_val:.5f}  (should be 1.0)')
print()

# Verify w = sum alpha_i * y_i * x_i  (using dual_coef_ = alpha_i * y_i)
w_dual = (clf_hard.dual_coef_ @ SVs).ravel()
print(f'  Verification via dual:  w = {w_dual.round(5)}')
print(f'  Matches coef_[0]:       w = {w_h.round(5)}')

## 2.2 Lagrangian Duality and the Dual Problem

We introduce **Lagrange multipliers** $\alpha_i \geq 0$ for each constraint and form the **Lagrangian**:
$$\mathcal{L}(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^n \alpha_i \bigl[y_i(\mathbf{w}\cdot\mathbf{x}_i + b) - 1\bigr].$$

**Stationarity conditions** (setting partial derivatives to zero):
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \mathbf{0}: \quad \mathbf{w} = \sum_{i=1}^n \alpha_i y_i \mathbf{x}_i.\qquad\qquad\frac{\partial \mathcal{L}}{\partial b} = 0: \quad \sum_{i=1}^n \alpha_i y_i = 0.$$

Substituting these back into $\mathcal{L}$ eliminates $\mathbf{w}$ and $b$, yielding the **dual problem**:

$$\boxed{\max_{\boldsymbol{\alpha} \geq 0}\; \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i,j=1}^n \alpha_i \alpha_j y_i y_j \langle \mathbf{x}_i, \mathbf{x}_j \rangle \qquad \text{subject to}\quad \sum_{i=1}^n \alpha_i y_i = 0.}$$

The dual is also a convex QP, now in the $n$ variables $\alpha_i$.

**KKT complementary slackness** requires $\alpha_i [y_i(\mathbf{w}\cdot\mathbf{x}_i+b)-1] = 0$ for all $i$. This means:
- If $\alpha_i > 0$, then $y_i(\mathbf{w}\cdot\mathbf{x}_i+b) = 1$ — the point lies exactly on the margin boundary (it is a **support vector**).
- If $y_i(\mathbf{w}\cdot\mathbf{x}_i+b) > 1$, then $\alpha_i = 0$ — the point is irrelevant.

**Critical observation:** The weight vector $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$ is a sparse linear combination of *support vectors only*, and the prediction function
$$f(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b = \sum_{i \in \mathrm{SV}} \alpha_i y_i \langle \mathbf{x}_i, \mathbf{x} \rangle + b$$
depends on the data **only through inner products** $\langle \mathbf{x}_i, \mathbf{x}_j \rangle$. This observation is the seed of the **kernel trick** (Part 4).

> **Exercise 2.1.** *(Deriving the dual)*
>
> **(a)** Starting from the Lagrangian above, carry out the substitution $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$ and $\sum_i \alpha_i y_i = 0$ in full detail to obtain the dual objective.
>
> **(b)** The dual problem is $\max_{\boldsymbol{\alpha}}\; \boldsymbol{\alpha}^\top \mathbf{1} - \tfrac{1}{2}\boldsymbol{\alpha}^\top Q \boldsymbol{\alpha}$, where $Q_{ij} = y_i y_j \langle \mathbf{x}_i, \mathbf{x}_j\rangle$.  Show that $Q$ is positive semidefinite (PSD).  Under what (geometric) condition on the data is $Q$ strictly positive definite?
>
> **(c)** Given the optimal dual solution $\boldsymbol{\alpha}^*$, derive a formula for the bias $b$ from any support vector $\mathbf{x}_s$ (i.e., any $i$ with $\alpha_i^* > 0$). In practice one averages over all support vectors for numerical stability; write this formula.
>
> **(d)** *(Complexity)* The primal has $d+1$ variables; the dual has $n$ variables.  If $d \gg n$ (e.g., text data: millions of features, thousands of documents), which formulation is computationally preferable?  What if $n \gg d$?

---
# Part 3: Soft-Margin SVM

## 3.1 When Data Is Not Linearly Separable

Real datasets are rarely linearly separable. Two pathologies arise:
1. **Infeasibility:** if no hyperplane separates the classes, the hard-margin QP has no solution.
2. **Sensitivity to outliers:** a single mislabeled point near the boundary can force an extremely narrow margin, hurting generalization.

The remedy is to *allow* some points to violate the margin, but penalize such violations.

## 3.2 Slack Variables and the $C$ Parameter

Introduce **slack variables** $\xi_i \geq 0$, one per training point. The **soft-margin (C-SVM) primal problem** is:
$$\min_{\mathbf{w},\, b,\, \boldsymbol{\xi}}\; \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^n \xi_i$$
$$\text{subject to}\quad y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 - \xi_i,\quad \xi_i \geq 0,\quad i=1,\ldots,n.$$

- $\xi_i = 0$: correctly classified and outside (or on) the margin.
- $0 < \xi_i < 1$: inside the margin but on the correct side.
- $\xi_i \geq 1$: misclassified.

The hyperparameter $C > 0$ controls the **bias–variance tradeoff**:
- **Large $C$**: penalize violations heavily $\Rightarrow$ try to correctly classify all training points $\Rightarrow$ narrow margin, risk of overfitting.
- **Small $C$**: allow more violations $\Rightarrow$ wide margin, more regularization, risk of underfitting.

The dual of the soft-margin SVM is identical to the hard-margin dual, except for an additional **box constraint**:
$$\max_{\boldsymbol{\alpha}}\; \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i\alpha_j y_i y_j\langle\mathbf{x}_i,\mathbf{x}_j\rangle \quad\text{s.t.}\quad 0 \leq \alpha_i \leq C,\quad \sum_i \alpha_i y_i = 0.$$

The upper bound $\alpha_i \leq C$ is the only difference. Points with $0 < \alpha_i < C$ are on the margin; those with $\alpha_i = C$ have $\xi_i > 0$ (they are in or past the margin).

In [ ]:
# Visualize the effect of C on the soft-margin SVM
rng2 = np.random.default_rng(42)
X_noisy_pos = rng2.multivariate_normal([1.0, 1.0],  [[1.1, 0.3], [0.3, 1.1]], size=60)
X_noisy_neg = rng2.multivariate_normal([-1.0, -1.0], [[1.1, 0.3], [0.3, 1.1]], size=60)
X_soft = np.vstack([X_noisy_pos, X_noisy_neg])
y_soft = np.hstack([np.ones(60), -np.ones(60)])

C_values = [0.01, 0.5, 5.0, 500.0]  # @param

fig, axes = plt.subplots(1, 4, figsize=(17, 4.8))
xx_bg = np.linspace(-4.5, 4.5, 250)
yy_bg = np.linspace(-4.5, 4.5, 250)
XX, YY = np.meshgrid(xx_bg, yy_bg)

for ax, C in zip(axes, C_values):
    clf_c = SVC(kernel='linear', C=C)
    clf_c.fit(X_soft, y_soft)

    Z = clf_c.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=[-200, -1, 0, 1, 200],
                colors=['#fde8e8', '#fde8e8', '#e8f4fd', '#e8f4fd'], alpha=0.5)
    ax.contour(XX, YY, Z, levels=[-1, 0, 1],
               colors=['#c0392b', 'k', '#2980b9'],
               linewidths=[1.5, 2.0, 1.5], linestyles=['--', '-', '--'])

    ax.scatter(X_noisy_pos[:, 0], X_noisy_pos[:, 1],
               c='#3498db', s=22, edgecolors='k', lw=0.3, zorder=3)
    ax.scatter(X_noisy_neg[:, 0], X_noisy_neg[:, 1],
               c='#e74c3c', s=22, marker='s', edgecolors='k', lw=0.3, zorder=3)

    sv_c = clf_c.support_vectors_
    ax.scatter(sv_c[:, 0], sv_c[:, 1], s=100, facecolors='none',
               edgecolors='gold', lw=2.0, zorder=4)

    w_c = clf_c.coef_[0]
    margin_c = 2.0 / np.linalg.norm(w_c)
    n_sv = len(sv_c)
    ax.set_xlim(-4.2, 4.2); ax.set_ylim(-4.2, 4.2)
    ax.set_title(f'C = {C}\nmargin = {margin_c:.2f},  {n_sv} SVs', fontsize=12)
    ax.set_xlabel(r'$x_1$')
    if ax is axes[0]:
        ax.set_ylabel(r'$x_2$')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(r'Effect of the regularization parameter $C$', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

> **Exercise 3.1.** *(The role of $C$ and the hinge loss)*
>
> **(a)** Run the visualization above with $C = 10^{-3}$, $10^{-1}$, $1$, $10^3$. Describe qualitatively how the margin width, number of support vectors, and number of misclassified training points change.
>
> **(b)** Show formally that as $C \to \infty$ (with linearly separable data), the soft-margin solution converges to the hard-margin solution.  *Hint:* What happens to the optimal $\xi_i$ as $C$ increases?
>
> **(c)** The soft-margin SVM can be rewritten in **unconstrained** form.  Show that the constrained problem
> $$\min_{\mathbf{w},b,\boldsymbol{\xi}}\; \tfrac{1}{2}\|\mathbf{w}\|^2 + C\textstyle\sum_i \xi_i \quad \text{s.t.} \quad y_i f(\mathbf{x}_i) \geq 1-\xi_i,\;\xi_i \geq 0$$
> is equivalent to:
> $$\min_{\mathbf{w},b}\; \frac{1}{2n}\|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^n \max\bigl(0,\, 1 - y_i f(\mathbf{x}_i)\bigr).$$
> The second term is the **hinge loss** $\ell_h(t) = \max(0, 1-t)$.
>
> **(d)** Plot $\ell_h(t)$, the logistic loss $\ell_\sigma(t) = \log(1+e^{-t})$, and the 0-1 loss $\ell_{0-1}(t) = \mathbf{1}[t < 0]$ on the same axes for $t \in [-2, 3]$. The hinge loss is zero for $t \geq 1$ (not just $t > 0$), while logistic loss is never exactly zero.  Explain why this difference implies that the SVM solution is *sparse* in the training data (few support vectors) while logistic regression uses all training points.

In [ ]:
# Plot and compare loss functions (Exercise 3.1d)
t = np.linspace(-2.2, 3.2, 400)
hinge   = np.maximum(0, 1 - t)
logistic = np.log1p(np.exp(-t))
zero_one = (t < 0).astype(float)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t, hinge,    lw=2.5, color='#e74c3c', label='Hinge: $\\max(0, 1-t)$')
ax.plot(t, logistic, lw=2.5, color='#3498db', label='Logistic: $\\log(1+e^{-t})$')
ax.plot(t, zero_one, lw=2.5, color='#2ecc71', linestyle='--',
        label='0\u20131 loss: $\\mathbf{1}[t<0]$')
ax.axvline(0, color='gray', lw=0.8, ls=':'); ax.axvline(1, color='gray', lw=0.8, ls=':')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('$t = y_i f(\\mathbf{x}_i)$ (margin)', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('Loss functions for binary classification', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-0.2, 3.0)
plt.tight_layout()
plt.show()

---
# Part 4: The Kernel Trick

## 4.1 The Limits of Linear Classifiers

Linear decision boundaries fail when classes are *intrinsically nonlinearly separable*.  The canonical example is the **XOR dataset**: points in quadrants I and III belong to one class; quadrants II and IV to the other.  No line in $\mathbb{R}^2$ can separate them.

In [ ]:
# XOR dataset and a failed linear SVM
rng_xor = np.random.default_rng(0)
n_xor = 120
X_xor = rng_xor.uniform(-1.2, 1.2, (n_xor, 2))
y_xor = np.sign(X_xor[:, 0] * X_xor[:, 1])
# Remove points very close to the axes (ambiguous region)
mask_xor = (np.abs(X_xor[:, 0]) > 0.12) & (np.abs(X_xor[:, 1]) > 0.12)
X_xor, y_xor = X_xor[mask_xor], y_xor[mask_xor]

# Try fitting a linear SVM
clf_lin_xor = SVC(kernel='linear', C=1.0)
clf_lin_xor.fit(X_xor, y_xor)
acc_lin = clf_lin_xor.score(X_xor, y_xor)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
pos_mask = y_xor == 1
neg_mask = y_xor == -1

for ax in axes:
    ax.scatter(X_xor[pos_mask, 0], X_xor[pos_mask, 1],
               c='#3498db', s=55, edgecolors='k', lw=0.5,
               label=r'$y=+1$ (quadrants I, III)', zorder=3)
    ax.scatter(X_xor[neg_mask, 0], X_xor[neg_mask, 1],
               c='#e74c3c', s=55, marker='s', edgecolors='k', lw=0.5,
               label=r'$y=-1$ (quadrants II, IV)', zorder=3)
    ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel(r'$x_1$', fontsize=13); ax.set_ylabel(r'$x_2$', fontsize=13)
    ax.legend(fontsize=10, loc='upper right')

# Left: data only
axes[0].set_title('XOR dataset (not linearly separable)', fontsize=13)

# Right: best-fit linear SVM boundary
xx2 = np.linspace(-1.4, 1.4, 300)
yy2 = np.linspace(-1.4, 1.4, 300)
XX2, YY2 = np.meshgrid(xx2, yy2)
Z_lin = clf_lin_xor.predict(np.c_[XX2.ravel(), YY2.ravel()]).reshape(XX2.shape)
axes[1].contourf(XX2, YY2, Z_lin, levels=[-2, 0, 2],
                 colors=['#fde8e8', '#e8f4fd'], alpha=0.4)
axes[1].contour(XX2, YY2, Z_lin, levels=[0], colors='k', linewidths=2)
axes[1].set_title(f'Best linear SVM (accuracy = {acc_lin:.2f})', fontsize=13)
# Replot data on top
axes[1].scatter(X_xor[pos_mask, 0], X_xor[pos_mask, 1],
                c='#3498db', s=55, edgecolors='k', lw=0.5, zorder=3)
axes[1].scatter(X_xor[neg_mask, 0], X_xor[neg_mask, 1],
                c='#e74c3c', s=55, marker='s', edgecolors='k', lw=0.5, zorder=3)

plt.tight_layout()
plt.show()
print(f'Linear SVM accuracy on XOR: {acc_lin:.2f}  (\u2248 chance level)')

## 4.2 Feature Maps: Lifting to Higher Dimensions

The standard remedy is to **embed** the data into a higher-dimensional space where the classes become linearly separable.

**Example.** For XOR data in $\mathbb{R}^2$, define the feature map:
$$\phi(x_1, x_2) = (x_1,\; x_2,\; x_1 x_2) \in \mathbb{R}^3.$$

In the lifted space, a hyperplane $\{\mathbf{w}\cdot\phi(\mathbf{x})+b=0\}$ can separate XOR classes because the product coordinate $x_1 x_2$ distinguishes the quadrants.  The cell below visualizes this.

**General approach:** Given $\phi: \mathbb{R}^d \to \mathbb{R}^D$ (possibly $D \gg d$, or even $D = \infty$), apply a linear SVM in the $\phi$-space:
$$f(\mathbf{x}) = \mathbf{w}\cdot\phi(\mathbf{x}) + b, \qquad \text{where}\quad \mathbf{w} \in \mathbb{R}^D.$$

**The catch:** Explicitly computing $\phi(\mathbf{x}) \in \mathbb{R}^D$ and working in $\mathbb{R}^D$ is expensive or impossible when $D$ is large.

In [ ]:
# Lift XOR data to 3D: phi(x1, x2) = (x1, x2, x1*x2)
phi_xor = np.column_stack([X_xor[:, 0], X_xor[:, 1],
                            X_xor[:, 0] * X_xor[:, 1]])

# Fit linear SVM in the lifted space
clf_lifted = SVC(kernel='linear', C=1e6)
clf_lifted.fit(phi_xor, y_xor)
w3 = clf_lifted.coef_[0]; b3 = clf_lifted.intercept_[0]
print(f'Linear SVM in lifted space accuracy: {clf_lifted.score(phi_xor, y_xor):.2f}')

fig = plt.figure(figsize=(14, 5.5))

# Left: 3D scatter + separating plane
ax3d = fig.add_subplot(121, projection='3d')
ax3d.scatter(phi_xor[pos_mask, 0], phi_xor[pos_mask, 1], phi_xor[pos_mask, 2],
             c='#3498db', s=35, edgecolors='k', lw=0.3, label='$y=+1$')
ax3d.scatter(phi_xor[neg_mask, 0], phi_xor[neg_mask, 1], phi_xor[neg_mask, 2],
             c='#e74c3c', s=35, marker='^', edgecolors='k', lw=0.3, label='$y=-1$')
# Separating hyperplane: z = -(w3[0]*x + w3[1]*y + b3) / w3[2]
xg, yg = np.meshgrid(np.linspace(-1.3, 1.3, 25), np.linspace(-1.3, 1.3, 25))
zg = (-w3[0]*xg - w3[1]*yg - b3) / w3[2]
zg = np.clip(zg, -2, 2)
ax3d.plot_surface(xg, yg, zg, alpha=0.25, color='gold', linewidth=0)
ax3d.set_xlabel(r'$x_1$'); ax3d.set_ylabel(r'$x_2$'); ax3d.set_zlabel(r'$x_1 x_2$')
ax3d.set_title(r'Lifted space: linearly separable!', fontsize=12)
ax3d.legend(fontsize=10, loc='upper left')

# Right: nonlinear boundary in original 2D space (via RBF SVM)
ax2d = fig.add_subplot(122)
clf_rbf_xor = SVC(kernel='rbf', C=10, gamma=5)
clf_rbf_xor.fit(X_xor, y_xor)

Z_rbf = clf_rbf_xor.predict(np.c_[XX2.ravel(), YY2.ravel()]).reshape(XX2.shape)
ax2d.contourf(XX2, YY2, Z_rbf, levels=[-2, 0, 2],
              colors=['#fde8e8', '#e8f4fd'], alpha=0.5)
ax2d.contour(XX2, YY2, Z_rbf, levels=[0], colors='k', linewidths=2)
ax2d.scatter(X_xor[pos_mask, 0], X_xor[pos_mask, 1],
             c='#3498db', s=50, edgecolors='k', lw=0.5, zorder=3)
ax2d.scatter(X_xor[neg_mask, 0], X_xor[neg_mask, 1],
             c='#e74c3c', s=50, marker='s', edgecolors='k', lw=0.5, zorder=3)
ax2d.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.4)
ax2d.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.4)
ax2d.set_xlabel(r'$x_1$', fontsize=13); ax2d.set_ylabel(r'$x_2$', fontsize=13)
ax2d.set_title(f'RBF SVM: nonlinear boundary in original space\n(acc = {clf_rbf_xor.score(X_xor, y_xor):.2f})',
               fontsize=12)

plt.tight_layout()
plt.show()

## 4.3 The Kernel Function

**Definition.** A **kernel** is a function $k: \mathbb{R}^d \times \mathbb{R}^d \to \mathbb{R}$ such that
$$k(\mathbf{x}, \mathbf{z}) = \langle \phi(\mathbf{x}),\, \phi(\mathbf{z}) \rangle$$
for some feature map $\phi: \mathbb{R}^d \to \mathcal{H}$ (where $\mathcal{H}$ is a Hilbert space, possibly infinite-dimensional).

**The kernel trick:** Replace every inner product $\langle \mathbf{x}_i, \mathbf{x}_j\rangle$ in the SVM dual with $k(\mathbf{x}_i, \mathbf{x}_j)$:

$$\max_{\boldsymbol{\alpha}}\; \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i,j=1}^n \alpha_i\alpha_j y_i y_j\, k(\mathbf{x}_i, \mathbf{x}_j) \quad\text{s.t.}\quad 0 \leq \alpha_i \leq C,\quad \sum_i \alpha_i y_i = 0.$$

The **prediction function** becomes:
$$f(\mathbf{x}) = \sum_{i \in \mathrm{SV}} \alpha_i y_i\, k(\mathbf{x}_i, \mathbf{x}) + b.$$

**We never compute $\phi(\mathbf{x})$ explicitly.** The only quantities we ever evaluate are pairwise kernel values $k(\mathbf{x}_i, \mathbf{x}_j)$.  If $k$ is cheap to evaluate, we can work in arbitrarily high (or infinite) dimensional spaces at the cost of an $n \times n$ kernel matrix.

The **kernel matrix** (Gram matrix) $K$ with $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ plays a central role: the dual objective in matrix form is $\boldsymbol{\alpha}^\top \mathbf{1} - \tfrac{1}{2}\boldsymbol{\alpha}^\top (K \odot \mathbf{y}\mathbf{y}^\top) \boldsymbol{\alpha}$, where $\odot$ denotes elementwise product.

## 4.4 Common Kernels

| Kernel | Formula $k(\mathbf{x},\mathbf{z})$ | Feature space dimension | Notes |
|--------|----------------------------------|------------------------|-------|
| **Linear** | $\mathbf{x}\cdot\mathbf{z}$ | $d$ | Identity map |
| **Polynomial (degree $p$)** | $(\mathbf{x}\cdot\mathbf{z}+c)^p$ | $\binom{d+p}{p}$ | $c>0$ adds lower-degree features |
| **RBF / Gaussian** | $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$ | $\infty$ | Most widely used |
| **Sigmoid** | $\tanh(\kappa\,\mathbf{x}\cdot\mathbf{z}+\theta)$ | — | Not always PSD |
| **Laplacian** | $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|_1)$ | $\infty$ | Sparser than RBF |

**RBF kernel parameters:**
- $\gamma > 0$: inverse bandwidth.  Large $\gamma$ $\Rightarrow$ each point influences only its immediate neighborhood (complex, possibly overfitting boundary).  Small $\gamma$ $\Rightarrow$ smooth, wide-influence boundary.
- `gamma='scale'` in scikit-learn sets $\gamma = 1/(d\cdot\mathrm{Var}(X))$; `'auto'` sets $\gamma = 1/d$.

**Polynomial kernel parameters:**
- $p$: degree.  Higher $p$ gives more expressive boundaries but risks overfitting.
- $c \geq 0$: controls the relative weight of lower- versus higher-degree terms.

> **Exercise 4.1.** *(Polynomial kernel and feature maps)*
>
> **(a)** Let $\mathbf{x}, \mathbf{z} \in \mathbb{R}^2$.  Expand $(\mathbf{x}\cdot\mathbf{z})^2 = (x_1 z_1 + x_2 z_2)^2$ and show that
> $$k(\mathbf{x},\mathbf{z}) = (\mathbf{x}\cdot\mathbf{z})^2 = \phi(\mathbf{x})\cdot\phi(\mathbf{z})$$
> with $\phi(x_1,x_2) = (x_1^2,\, \sqrt{2}\,x_1 x_2,\, x_2^2)$.
>
> **(b)** Generalize: what is the dimension of the feature space for the homogeneous polynomial kernel $(\mathbf{x}\cdot\mathbf{z})^p$ on $\mathbb{R}^d$?  What about the inhomogeneous kernel $(\mathbf{x}\cdot\mathbf{z}+1)^p$?
>
> **(c)** The RBF kernel can be written $k(\mathbf{x},\mathbf{z}) = e^{-\gamma\|\mathbf{x}\|^2}\, e^{-\gamma\|\mathbf{z}\|^2}\, e^{2\gamma \mathbf{x}\cdot\mathbf{z}}$. Using the Taylor expansion $e^t = \sum_{n=0}^\infty t^n/n!$, show that
> $$e^{-\gamma\|\mathbf{x}-\mathbf{z}\|^2} = \sum_{n=0}^\infty \frac{(2\gamma)^n}{n!}\,e^{-\gamma\|\mathbf{x}\|^2}\,e^{-\gamma\|\mathbf{z}\|^2}\,(\mathbf{x}\cdot\mathbf{z})^n.$$
> Conclude that the RBF kernel corresponds to an **infinite-dimensional** feature map — a weighted superposition of all polynomial feature maps.
>
> **(d)** *(Computation)* Using the RBF kernel with $\gamma=0.5$ and NumPy broadcasting, compute the $6\times 6$ kernel matrix for the six points in Exercise 1.1 and verify it is symmetric and positive definite (check eigenvalues).

In [ ]:
# Compare four kernels across three datasets
from sklearn.preprocessing import StandardScaler

datasets = [
    ('Circles',   *make_circles(n_samples=200, noise=0.10, factor=0.4, random_state=1)),
    ('Moons',     *make_moons(  n_samples=200, noise=0.18, random_state=1)),
    ('XOR',        X_xor,  y_xor),
]

kernel_specs = [
    ('Linear',       dict(kernel='linear',                   C=1.0)),
    ('RBF',          dict(kernel='rbf',    gamma='scale',    C=1.0)),
    ('Poly (deg 3)', dict(kernel='poly',   degree=3, coef0=1, C=1.0)),
    ('Poly (deg 5)', dict(kernel='poly',   degree=5, coef0=1, C=1.0)),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))

for row, (ds_name, X_ds, y_ds) in enumerate(datasets):
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_ds)
    x_min, x_max = X_sc[:, 0].min() - 0.5, X_sc[:, 0].max() + 0.5
    y_min, y_max = X_sc[:, 1].min() - 0.5, X_sc[:, 1].max() + 0.5
    xxg = np.linspace(x_min, x_max, 250)
    yyg = np.linspace(y_min, y_max, 250)
    XXg, YYg = np.meshgrid(xxg, yyg)

    for col, (k_name, params) in enumerate(kernel_specs):
        ax = axes[row, col]
        clf_k = SVC(**params)
        clf_k.fit(X_sc, y_ds)
        acc = clf_k.score(X_sc, y_ds)

        Zk = clf_k.predict(np.c_[XXg.ravel(), YYg.ravel()]).reshape(XXg.shape)
        ax.contourf(XXg, YYg, Zk, levels=[-2, 0, 2],
                    colors=['#fde8e8', '#e8f4fd'], alpha=0.5)
        ax.contour(XXg, YYg, Zk, levels=[0], colors='k', linewidths=1.8)
        ax.scatter(X_sc[y_ds ==  1, 0], X_sc[y_ds ==  1, 1],
                   c='#3498db', s=20, edgecolors='k', lw=0.3, zorder=3)
        ax.scatter(X_sc[y_ds != 1, 0],  X_sc[y_ds != 1, 1],
                   c='#e74c3c', s=20, marker='s', edgecolors='k', lw=0.3, zorder=3)
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{k_name}\nacc = {acc:.2f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=12, fontweight='bold')

plt.suptitle('Kernel comparison across datasets', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4.5 The $\gamma$ Parameter of the RBF Kernel — Interactive Exploration

The RBF kernel $k(\mathbf{x},\mathbf{z}) = e^{-\gamma\|\mathbf{x}-\mathbf{z}\|^2}$ has a clear geometric interpretation: it measures *similarity* as a function of Euclidean distance, with $\gamma$ controlling the scale.  The cell below lets you explore the joint effect of $C$ and $\gamma$ on the decision boundary.  Adjust the parameters and re-run.

**Experiment:** Try the extremes — $C=0.05, \gamma=0.5$ (underfit) versus $C=100, \gamma=20$ (overfit).  What does each boundary look like?

In [ ]:
# Interactive RBF exploration on the moons dataset
# Change these values and re-run the cell
C_rbf    = 1.0    # @param {type:"number"}
gamma_rbf = 1.0   # @param {type:"number"}

X_moon, y_moon = make_moons(n_samples=180, noise=0.22, random_state=42)
scaler_moon = StandardScaler()
X_moon_s = scaler_moon.fit_transform(X_moon)

clf_rbf = SVC(kernel='rbf', C=C_rbf, gamma=gamma_rbf)
clf_rbf.fit(X_moon_s, y_moon)
train_acc = clf_rbf.score(X_moon_s, y_moon)

xm = np.linspace(X_moon_s[:, 0].min()-0.4, X_moon_s[:, 0].max()+0.4, 350)
ym = np.linspace(X_moon_s[:, 1].min()-0.4, X_moon_s[:, 1].max()+0.4, 350)
XXm, YYm = np.meshgrid(xm, ym)
Zm = clf_rbf.decision_function(np.c_[XXm.ravel(), YYm.ravel()]).reshape(XXm.shape)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.contourf(XXm, YYm, Zm, levels=20, cmap='RdBu_r', alpha=0.65)
ax.contour(XXm, YYm, Zm, levels=[-1, 0, 1],
           colors=['#c0392b', 'k', '#2980b9'],
           linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])
ax.scatter(X_moon_s[y_moon==0, 0], X_moon_s[y_moon==0, 1],
           c='#e74c3c', s=50, edgecolors='k', lw=0.5, zorder=3)
ax.scatter(X_moon_s[y_moon==1, 0], X_moon_s[y_moon==1, 1],
           c='#3498db', s=50, edgecolors='k', lw=0.5, zorder=3)
sv_m = clf_rbf.support_vectors_
ax.scatter(sv_m[:, 0], sv_m[:, 1], s=130, facecolors='none',
           edgecolors='gold', lw=2.2, zorder=4, label=f'{len(sv_m)} SVs')
plt.colorbar(im, ax=ax, label='Decision score')
ax.set_xlabel(r'$x_1$ (scaled)', fontsize=13)
ax.set_ylabel(r'$x_2$ (scaled)', fontsize=13)
ax.set_title(f'RBF SVM   C={C_rbf}, $\\gamma$={gamma_rbf}\n'
             f'Train accuracy = {train_acc:.3f}', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 4.6 Mercer's Theorem and Valid Kernels

Not every symmetric function $k: \mathbb{R}^d \times \mathbb{R}^d \to \mathbb{R}$ corresponds to an inner product in some feature space.  Mercer's theorem characterizes valid kernels.

**Mercer's theorem.**  A continuous symmetric function $k$ is a valid kernel (i.e., $k(\mathbf{x},\mathbf{z}) = \langle\phi(\mathbf{x}),\phi(\mathbf{z})\rangle_{\mathcal{H}}$ for some Hilbert space $\mathcal{H}$ and map $\phi$) if and only if the **kernel matrix** $K$ with $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ is **positive semidefinite (PSD)** for every finite set of points $\{\mathbf{x}_1,\ldots,\mathbf{x}_n\}$.

Equivalently: $k$ is a valid kernel iff $\sum_{i,j} c_i c_j k(\mathbf{x}_i, \mathbf{x}_j) \geq 0$ for all $n$, all $\mathbf{x}_1,\ldots,\mathbf{x}_n$, and all $c_1,\ldots,c_n \in \mathbb{R}$.

**Kernel composition rules** (for constructing new kernels from old ones):

| Construction | Result |
|---|---|
| $k_1 + k_2$ | valid kernel |
| $k_1 \cdot k_2$ | valid kernel |
| $c \cdot k_1$, $c > 0$ | valid kernel |
| $f(\mathbf{x})\,k_1(\mathbf{x},\mathbf{z})\,f(\mathbf{z})$ for any $f$ | valid kernel |
| $\exp(k_1(\mathbf{x},\mathbf{z}))$ | valid kernel |
| $p(k_1(\mathbf{x},\mathbf{z}))$ for polynomial $p$ with nonneg. coefficients | valid kernel |

These rules explain why the RBF kernel is valid: $e^{-\gamma\|\mathbf{x}-\mathbf{z}\|^2} = e^{-\gamma\|\mathbf{x}\|^2}\cdot e^{2\gamma \mathbf{x}\cdot\mathbf{z}} \cdot e^{-\gamma\|\mathbf{z}\|^2}$, which combines the composition rules above applied to the linear (hence valid) kernel.

In [ ]:
# Visualize the RBF kernel as a similarity function
gamma_vals = [0.2, 1.0, 5.0]
x_center = np.array([0.0])
xplot = np.linspace(-3, 3, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: 1D kernel profiles
for g in gamma_vals:
    k_vals = np.exp(-g * (xplot - x_center[0])**2)
    axes[0].plot(xplot, k_vals, lw=2.5, label=f'$\\gamma={g}$')
axes[0].axvline(0, color='gray', lw=0.8, ls='--')
axes[0].set_xlabel(r'$x - x_0$', fontsize=13)
axes[0].set_ylabel(r'$k(x_0, x)$', fontsize=13)
axes[0].set_title(r'RBF kernel profile centered at $x_0=0$', fontsize=13)
axes[0].legend(fontsize=11)

# Right: 2D kernel heatmap (gamma = 1)
x0 = np.array([0.0, 0.0])
xg2, yg2 = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
pts = np.c_[xg2.ravel(), yg2.ravel()]
K2d = np.exp(-1.0 * np.sum((pts - x0)**2, axis=1)).reshape(xg2.shape)
im2 = axes[1].contourf(xg2, yg2, K2d, levels=20, cmap='YlOrRd')
plt.colorbar(im2, ax=axes[1], label=r'$k(\mathbf{x}_0, \mathbf{x})$')
axes[1].plot(*x0, 'k*', ms=12, label=r'$\mathbf{x}_0 = (0,0)$')
axes[1].set_xlabel(r'$x_1$', fontsize=13); axes[1].set_ylabel(r'$x_2$', fontsize=13)
axes[1].set_title(r'RBF kernel $k(\mathbf{x}_0, \mathbf{x})$ for $\gamma=1$', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

> **Exercise 4.2.** *(Custom kernels and Mercer's theorem)*
>
> **(a)** For each of the following functions, determine whether it is a valid Mercer kernel, and justify your answer (by exhibiting a feature map, applying composition rules, or finding a counterexample):
> 1. $k(\mathbf{x},\mathbf{z}) = (\mathbf{x}\cdot\mathbf{z})^3 - 2$
> 2. $k(\mathbf{x},\mathbf{z}) = \|\mathbf{x}-\mathbf{z}\|^2$
> 3. $k(\mathbf{x},\mathbf{z}) = \exp(-\|\mathbf{x}-\mathbf{z}\|^2) + 0.5\,(\mathbf{x}\cdot\mathbf{z})^2$
>
> **(b)** *(String kernel)* Let $\mathcal{A}$ be an alphabet (e.g., $\mathcal{A}=\{A,C,G,T\}$).  For two strings $s, t \in \mathcal{A}^*$, define the **spectrum kernel** of length $\ell$:
> $$k_\ell(s, t) = \sum_{u \in \mathcal{A}^\ell} \phi_u(s)\,\phi_u(t),$$
> where $\phi_u(s)$ counts the number of occurrences of substring $u$ in $s$.
> Show that $k_\ell$ is a valid Mercer kernel.
>
> **(c)** Implement $k_2$ (all substrings of length 2) in Python.  Compute the kernel matrix for the strings `["ACGT", "ACGA", "TTGA", "TTGT"]` and verify all eigenvalues are nonneg.

In [ ]:
# Exercise 4.2c scaffold: spectrum kernel of length 2
from itertools import product as iproduct

def spectrum_kernel(s1, s2, ell=2):
    """Count shared substrings of length ell between s1 and s2."""
    def get_substrings(s, l):
        return [s[i:i+l] for i in range(len(s) - l + 1)]
    sub1 = get_substrings(s1, ell)
    sub2 = get_substrings(s2, ell)
    # Count shared substrings (= dot product of count vectors)
    vocab = set(sub1) | set(sub2)
    return sum(sub1.count(v) * sub2.count(v) for v in vocab)

sequences = ['ACGT', 'ACGA', 'TTGA', 'TTGT']
n_seq = len(sequences)
K_spectrum = np.array([[spectrum_kernel(sequences[i], sequences[j])
                        for j in range(n_seq)] for i in range(n_seq)], dtype=float)

print('Spectrum kernel matrix (ell=2):')
print(K_spectrum)

eigvals = np.linalg.eigvalsh(K_spectrum)
print(f'\nEigenvalues: {eigvals.round(6)}')
print(f'All nonneg (PSD): {np.all(eigvals >= -1e-10)}')

---
# Part 5: SVMs in Practice

## 5.1 Hyperparameter Selection

For kernel SVMs, two parameters must be tuned jointly:
- **$C$**: controls the margin–violation tradeoff (applies to all kernels).
- **$\gamma$**: controls the RBF kernel bandwidth (or polynomial degree/coefficient).

Both parameters span many orders of magnitude; a **logarithmic grid search** with **cross-validation** is standard practice.  The code below runs a $4\times4$ grid search and visualizes the CV accuracy landscape.

In [ ]:
# Grid search: C and gamma for RBF SVM on moons dataset
X_gs, y_gs = make_moons(n_samples=300, noise=0.25, random_state=42)

pipe_gs = Pipeline([('scaler', StandardScaler()), ('svc', SVC(kernel='rbf'))])
param_grid = {'svc__C':     [0.1, 1.0, 10.0, 100.0],
              'svc__gamma': [0.01, 0.1, 1.0, 10.0]}
grid_search = GridSearchCV(pipe_gs, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_gs, y_gs)

scores = grid_search.cv_results_['mean_test_score'].reshape(4, 4)
C_vals  = [0.1, 1.0, 10.0, 100.0]
gm_vals = [0.01, 0.1, 1.0, 10.0]

fig, (ax_heat, ax_bound) = plt.subplots(1, 2, figsize=(14, 5.5))

# Heatmap of CV scores
im = ax_heat.imshow(scores, cmap='viridis',
                    vmin=scores.min(), vmax=min(scores.max()+0.02, 1.0), aspect='auto')
ax_heat.set_xticks(range(4)); ax_heat.set_xticklabels(gm_vals)
ax_heat.set_yticks(range(4)); ax_heat.set_yticklabels(C_vals)
ax_heat.set_xlabel(r'$\gamma$', fontsize=13)
ax_heat.set_ylabel('$C$', fontsize=13)
ax_heat.set_title('5-fold CV accuracy (RBF SVM)', fontsize=13)
for i in range(4):
    for j in range(4):
        ax_heat.text(j, i, f'{scores[i,j]:.3f}', ha='center', va='center', fontsize=10,
                     color='white' if scores[i,j] < scores.mean() else 'black')
plt.colorbar(im, ax=ax_heat)

# Best model decision boundary
best = grid_search.best_estimator_
X_gs_s = best.named_steps['scaler'].transform(X_gs)
x_lo, x_hi = X_gs_s[:,0].min()-0.4, X_gs_s[:,0].max()+0.4
y_lo, y_hi = X_gs_s[:,1].min()-0.4, X_gs_s[:,1].max()+0.4
xxb, yyb = np.meshgrid(np.linspace(x_lo, x_hi, 300), np.linspace(y_lo, y_hi, 300))
Zb = best.named_steps['svc'].decision_function(
         np.c_[xxb.ravel(), yyb.ravel()]).reshape(xxb.shape)

ax_bound.contourf(xxb, yyb, Zb, levels=20, cmap='RdBu_r', alpha=0.65)
ax_bound.contour(xxb, yyb, Zb, levels=[0], colors='k', linewidths=2)
ax_bound.scatter(X_gs_s[y_gs==0, 0], X_gs_s[y_gs==0, 1],
                 c='#e74c3c', s=25, edgecolors='k', lw=0.3, zorder=3)
ax_bound.scatter(X_gs_s[y_gs==1, 0], X_gs_s[y_gs==1, 1],
                 c='#3498db', s=25, edgecolors='k', lw=0.3, zorder=3)
bc = grid_search.best_params_['svc__C']
bg = grid_search.best_params_['svc__gamma']
bs = grid_search.best_score_
ax_bound.set_xlabel(r'$x_1$ (scaled)', fontsize=13)
ax_bound.set_ylabel(r'$x_2$ (scaled)', fontsize=13)
ax_bound.set_title(f'Best model: C={bc}, $\\gamma$={bg}\nCV accuracy = {bs:.3f}', fontsize=13)

plt.tight_layout()
plt.show()
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

## 5.2 Multi-Class Classification

SVMs are fundamentally binary classifiers.  For $K > 2$ classes, two standard strategies extend SVMs:

**One-vs-Rest (OvR) / One-vs-All:** Train $K$ binary classifiers.  Classifier $k$ distinguishes class $k$ from all other classes.  Prediction: $\hat{y} = \arg\max_k f_k(\mathbf{x})$ (the class with the highest decision score).  Default in `LinearSVC`.

**One-vs-One (OvO):** Train $\binom{K}{2}$ binary classifiers, one per pair of classes.  Each classifier "votes"; the class with the most votes wins.  Default in `SVC`.

| Strategy | # classifiers | Training set size per clf | Notes |
|----------|:---:|:---:|-------|
| OvR | $K$ | $n$ | Fast; unbalanced subproblems |
| OvO | $K(K-1)/2$ | $\approx 2n/K$ | Slower for large $K$; balanced subproblems |

In [ ]:
# Multi-class SVM on the Iris dataset (all 4 features)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

scaler_iris = StandardScaler()
X_tr_s = scaler_iris.fit_transform(X_tr)
X_te_s = scaler_iris.transform(X_te)

kernels_iris = ['linear', 'rbf', 'poly']
print(f'{"Kernel":<12} {"Train acc":>10} {"Test acc":>10} {"# SVs":>8}')
print('-' * 44)
for k in kernels_iris:
    clf_iris = SVC(kernel=k, C=1.0, gamma='scale', degree=3)
    clf_iris.fit(X_tr_s, y_tr)
    tr_acc = clf_iris.score(X_tr_s, y_tr)
    te_acc = clf_iris.score(X_te_s, y_te)
    n_sv = clf_iris.n_support_.sum()
    print(f'{k:<12} {tr_acc:>10.4f} {te_acc:>10.4f} {n_sv:>8}')

In [ ]:
# Visualize multi-class decision boundaries (using petal features for 2D plot)
feat_idx = [2, 3]  # petal length, petal width
X_iris_2d = X_iris[:, feat_idx]
feat_names = [iris.feature_names[i] for i in feat_idx]
class_names = iris.target_names
cols_iris = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

for ax, k in zip(axes, kernels_iris):
    sc2 = StandardScaler()
    X2s = sc2.fit_transform(X_iris_2d)
    clf2 = SVC(kernel=k, C=1.0, gamma='scale', degree=3)
    clf2.fit(X2s, y_iris)
    acc2 = clf2.score(X2s, y_iris)

    x_lo2, x_hi2 = X2s[:,0].min()-0.4, X2s[:,0].max()+0.4
    y_lo2, y_hi2 = X2s[:,1].min()-0.4, X2s[:,1].max()+0.4
    xx2d = np.linspace(x_lo2, x_hi2, 350)
    yy2d = np.linspace(y_lo2, y_hi2, 350)
    XX2d, YY2d = np.meshgrid(xx2d, yy2d)
    Z2d = clf2.predict(np.c_[XX2d.ravel(), YY2d.ravel()]).reshape(XX2d.shape)

    cmap_bg = ListedColormap(['#fde8e8', '#d5f5e3', '#d6eaf8'])
    ax.contourf(XX2d, YY2d, Z2d, cmap=cmap_bg, alpha=0.65)
    ax.contour(XX2d, YY2d, Z2d, colors='k', linewidths=0.9, alpha=0.5)

    for cls_idx, (cls, col) in enumerate(zip(class_names, cols_iris)):
        mask_cls = y_iris == cls_idx
        ax.scatter(X2s[mask_cls, 0], X2s[mask_cls, 1],
                   c=col, s=40, edgecolors='k', lw=0.4, label=cls, zorder=3)

    ax.set_xlabel(feat_names[0], fontsize=11)
    ax.set_ylabel(feat_names[1], fontsize=11)
    ax.set_title(f'Kernel: {k}  (acc = {acc2:.2f})', fontsize=12)
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle('Iris: Multi-class SVM (OvO strategy)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

> **Exercise 5.1.** *(Comparing classifiers)*
>
> **(a)** Using `make_moons(n_samples=300, noise=0.25, random_state=42)` with an 80/20 train/test split, train the following classifiers and compare their **test** accuracy:
> - Linear SVM (`kernel='linear'`, `C=1.0`)
> - RBF SVM (`C=1.0, gamma='scale'`)
> - Polynomial SVM (degree 3, `coef0=1`, `C=1.0`)
> - Logistic Regression (`sklearn.linear_model.LogisticRegression`, `C=1.0`)
>
> **(b)** For the RBF SVM, perform a grid search over $C \in \{0.1, 1, 10, 100\}$ and $\gamma \in \{0.01, 0.1, 1, 10\}$ using 5-fold CV on the **training set only**.  Report the best parameters and their test accuracy.
>
> **(c)** Plot the decision boundaries of the best SVM and logistic regression side by side on the **test set**.  Comment on the qualitative differences.
>
> **(d)** *(Graduate level — Representer Theorem)*  The representer theorem states: for any regularized ERM problem of the form
> $$\min_{f \in \mathcal{H}_k} \frac{\lambda}{2}\|f\|_{\mathcal{H}_k}^2 + \frac{1}{n}\sum_{i=1}^n L(y_i, f(\mathbf{x}_i)),$$
> the minimizer can be written as $f^*(\mathbf{x}) = \sum_{i=1}^n \alpha_i k(\mathbf{x}_i,\mathbf{x})$ for some $\boldsymbol{\alpha} \in \mathbb{R}^n$, regardless of the loss $L$.  
> Prove this result: show that the component of $f$ orthogonal to $\mathrm{span}\{k(\mathbf{x}_i,\cdot)\}$ does not affect the loss terms but only increases the norm, so the optimal $f$ has no such component.  *Hint:* decompose $f = f_{\parallel} + f_{\perp}$ and use the reproducing property $f(\mathbf{x}_i) = \langle f, k(\mathbf{x}_i,\cdot)\rangle_{\mathcal{H}_k}$.

---
# Summary

Support Vector Machines offer a principled approach to classification grounded in geometric intuition, convex optimization, and functional analysis.

| Concept | Key Idea |
|---|---|
| **Maximum margin** | Among all separating hyperplanes, choose the one with largest distance to nearest points |
| **Support vectors** | Only the $\alpha_i > 0$ training points determine the classifier; all others can be removed |
| **Hard-margin SVM** | Assumes linear separability; minimize $\frac{1}{2}\|\mathbf{w}\|^2$ s.t. $y_i(\mathbf{w}\cdot\mathbf{x}_i+b)\geq 1$ |
| **Soft-margin SVM** | Allows margin violations; $C$ trades off margin width vs. misclassification penalty |
| **Lagrangian dual** | QP in $\alpha_i \in [0,C]$; data appears only through inner products $\langle\mathbf{x}_i,\mathbf{x}_j\rangle$ |
| **Kernel trick** | Replace $\langle\mathbf{x}_i,\mathbf{x}_j\rangle \to k(\mathbf{x}_i,\mathbf{x}_j)$; work implicitly in high-dim. feature space |
| **RBF kernel** | Corresponds to infinite-dim. feature space; $\gamma$ controls locality / boundary smoothness |
| **Mercer's theorem** | $k$ is a valid kernel $\iff$ the Gram matrix $K$ is PSD for every finite dataset |
| **Multi-class** | OvO (default in `SVC`) or OvR (default in `LinearSVC`) |

**When are SVMs a good choice?**
- Medium-sized datasets (thousands to tens of thousands of samples).
- High feature dimension relative to sample count (e.g., text, genomics).
- When a good kernel is available (images, strings, graphs, time series).
- When interpretability via support vectors is valuable.

**Further reading:**
- B. Schölkopf & A. Smola, *Learning with Kernels* (2002) — the authoritative reference.
- C. M. Bishop, *Pattern Recognition and Machine Learning*, Chapter 7.
- V. N. Vapnik, *The Nature of Statistical Learning Theory*, 2nd ed. (2000).
- C.-C. Chang & C.-J. Lin, "LIBSVM: A library for support vector machines," *ACM TIST*, 2011.